# 05 Stacking: the family model's output as features for a binary model

**Question:** can a binary model do better if it also sees what the **family-first
model** thinks of each domain?

Two levels:

| Level | Model | Input | Output |
|---|---|---|---|
| 1 | family-first logistic regression | the hand-crafted features | P(legit), P(family 1), …, P(family 103) |
| 2 | binary logistic regression | features **+** the level-1 probabilities | DGA / legit |

We compare four methods on the same training samples and the same validation set:

| Method | What it is |
|---|---|
| `binary` | binary model on the features (as in notebook 04) |
| `family` | family-first, P(DGA) = 1 − P(legit) (as in notebook 04) |
| `stack_probs` | binary model on the family probabilities **only** |
| `stack_both` | binary model on the features **+** the family probabilities |

The code is in `src/models/stacking.py`.

### Avoiding leakage (out-of-fold)

If the family model is trained on a domain and then gives its probabilities for **that same
domain**, they are too confident, and the binary model learns to trust them too much. On new
domains they are less reliable, so the result would be too optimistic.

So, as for the n-gram features:

- **training rows**: the training sample is cut into 5 folds; each fold gets its
  probabilities from a family model trained on the **other 4 folds**;
- **validation rows**: family model trained on the **whole** training sample.

This means 6 family fits per seed and TLD setting (5 folds + 1 full), so this notebook is
slower than notebook 04.

## Settings

- `SEEDS`: training samples; seeds 0, 1, 2 are the same 100k samples as in notebook 04,
  so `binary` and `family` should give the same numbers as there (a check).
- `N_FOLDS`: folds for the out-of-fold probabilities.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

SAMPLE = 100_000   # same as notebook 04
SEEDS = range(3)   # 3 seeds to keep the run time reasonable
N_FOLDS = 5
EVAL_SET = "val"

## Data and features

The team's split (`load_splits()`) and the 74 features (`build_features()`, from the cache).

In [2]:
import pandas as pd

from src.data.dataset import load_splits
from src.features.build import build_features

train_full, val, test = load_splits()
feats = build_features(train_full, val, test)

evaluate = test if EVAL_SET == "test" else val
X_eval_all = feats["test"] if EVAL_SET == "test" else feats["val"]
print(f"features: {len(feats['name_cols'])} name + {len(feats['tld_cols'])} TLD")

features: 20 name + 54 TLD


## Train and test

For every seed and TLD setting: level 1 (out-of-fold family probabilities), then the four
methods. Everything is logged in MLflow (experiment `baselines`, runs `logreg_stacking ...`).

In [3]:
from src.models.stacking import run_stacking, summarize_stacking

table = run_stacking(feats, train_full, evaluate, X_eval_all, seeds=SEEDS, sample=SAMPLE,
                     n_folds=N_FOLDS, eval_set=EVAL_SET)
summary, gains = summarize_stacking(table)

seed 0 | TLD yes ...
seed 0 | TLD no ...
seed 1 | TLD yes ...
seed 1 | TLD no ...
seed 2 | TLD yes ...
seed 2 | TLD no ...


## Results: macro-F1 per method

Mean ± std over the seeds, on validation. Check: `binary` and `family` should match notebook 04.

In [4]:
(summary["mean"].round(4).astype(str) + " ± " + summary["std"].round(4).astype(str)).unstack("TLD")

TLD,no,yes
method,,
binary,0.9089 ± 0.0002,0.9211 ± 0.0001
family,0.9149 ± 0.0005,0.9395 ± 0.0002
stack_both,0.9136 ± 0.0003,0.9349 ± 0.0003
stack_probs,0.9138 ± 0.0003,0.9352 ± 0.0003


## Results: do the stacked models beat the references?

Gain = stacked − reference macro-F1, per seed, then averaged. "better" = mean gain larger
than its standard deviation (the proposal's rule).

In [5]:
gains.round(4)

mean_gain  std_gain better
TLD comparison                                     
no  stack_probs - binary  0.004904  0.000115   True
    stack_probs - family -0.001127  0.000221  False
    stack_both - binary   0.004677  0.000162   True
    stack_both - family  -0.001354  0.000206  False
yes stack_probs - binary  0.014081  0.000201   True
    stack_probs - family -0.004312  0.000407  False
    stack_both - binary   0.013774  0.000216   True
    stack_both - family  -0.004619  0.000348  False

**Stacking result.** Stacking beats binary (+0.0141 with TLD, +0.0049 without, gain > std)
but does not beat plain family-first (−0.0043 with TLD, −0.0011 without). stack_probs and
stack_both are almost equal: the family probabilities already carry the information of the
features. Conclusion: the best way to use the families is the simple collapse P(DGA) = 1 − P(legit).